In [ ]:
## 概述

延伸 `Day3/3-1` 的便利商店購物車練習，加入訂單 CSV 檔案、資料夾管理與基本例外處理。程式執行時不需要輸入 Linux 指令，而是使用 Python 操作作業系統中的資料夾與檔案。

---

## 情境說明

你正在製作一個便利商店自助結帳系統。顧客可以把商品加入購物車、查看購物車、移除商品，最後結帳產生訂單檔案。

訂單檔案會儲存在 `orders` 資料夾中。程式需能列出目前有哪些訂單 CSV，並讀取指定訂單顯示在螢幕上。
## 本題會練習到的語法

- `list`
- `dict`
- `if / elif / else`
- `while`
- `for`
- `print`
- `f-string`
- `try-except`
- `with open`
- `csv`
- `os.path.exists`
- `os.makedirs`
- `os.listdir`
- `datetime`

## 可使用的套件

以下都是 Python 內建套件，不需要額外安裝。

```python
import csv
import os
from datetime import datetime
```

In [1]:
import csv
import os
from datetime import datetime

# 商品目錄
catalog = {
    "御飯糰": 35,
    "礦泉水": 20,
    "布丁": 25,
    "關東煮": 15,
    "飯糰": 30,
}

# 購物車資料結構
cart = {}

# 訂單資料夾名稱
ORDER_DIR = "orders"

# 初始化：檢查並建立訂單資料夾
try:
    if not os.path.exists(ORDER_DIR):
        os.makedirs(ORDER_DIR)
        print(f"已建立 {ORDER_DIR} 資料夾。")
except Exception as e:
    print(f"建立資料夾失敗: {e}")

def display_menu():
    print("\n歡迎使用便利商店自助結帳系統！")
    print("請選擇操作：")
    print("1. 加入商品到購物車")
    print("2. 查看購物車")
    print("3. 移除購物車中的商品")
    print("4. 結帳")
    print("5. 列出訂單")
    print("6. 查看訂單內容")
    print("7. 離開")

def add_to_cart():
    print("\n商品目錄：")
    for item, price in catalog.items():
        print(f"{item}: {price} 元")
    
    item_name = input("請輸入要加入購物車的商品名稱：")
    if item_name in catalog:
        try:
            quantity = int(input("請輸入數量："))
            if quantity > 0:
                if item_name in cart:
                    cart[item_name] += quantity
                else:
                    cart[item_name] = quantity
                print(f"{quantity} 個 {item_name} 已加入購物車。")
            else:
                print("數量必須大於 0。")
        except ValueError:
            print("請輸入有效的數字。")
    else:
        print("商品不存在，請重新輸入。")

def view_cart():
    if cart:
        print("\n購物車內容：")
        total = 0
        for item, quantity in cart.items():
            price = catalog[item]
            subtotal = price * quantity
            total += subtotal
            print(f"{item} x {quantity} = {subtotal} 元")
        print(f"總計: {total} 元")
    else:
        print("\n購物車是空的。")

def remove_from_cart():
    if cart:
        print("\n購物車內容：")
        for item, quantity in cart.items():
            print(f"{item} x {quantity}")
        
        item_name = input("請輸入要移除的商品名稱：")
        if item_name in cart:
            del cart[item_name]
            print(f"{item_name} 已從購物車移除。")
        else:
            print("商品不在購物車中，請重新輸入。")
    else:
        print("\n購物車是空的。")

def checkout():
    if cart:    
        total = sum(catalog[item] * quantity for item, quantity in cart.items())
        print(f"\n您的總計是 {total} 元。")
        confirm = input("是否要結帳？(y/n)：")
        if confirm.lower() == 'y':
            # 使用當前時間作為訂單 ID
            order_id = datetime.now().strftime("%Y%m%d%H%M%S")
            order_file = os.path.join(ORDER_DIR, f"order_{order_id}.csv")
            try:
                with open(order_file, mode='w', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    # 寫入標題
                    writer.writerow(["商品名稱", "數量", "單價", "小計"])
                    # 寫入商品品項
                    for item, quantity in cart.items():
                        price = catalog[item]
                        subtotal = price * quantity
                        writer.writerow([item, quantity, price, subtotal])
                    # 寫入總計
                    writer.writerow(["總計", "", "", total])
                
                print(f"訂單已儲存為 {order_file}。")
                cart.clear() # 結帳完成後清空購物車
            except Exception as e:
                print(f"儲存訂單失敗: {e}")
        else:
            print("結帳已取消。")
    else:
        print("\n購物車是空的，無法結帳。")

def list_orders():
    try:
        orders = os.listdir(ORDER_DIR)
        if orders:
            print("\n目前的訂單：")
            for order in orders:
                print(order)
        else:
            print("\n目前沒有訂單。")
    except Exception as e:
        print(f"讀取訂單失敗: {e}")

def view_order():
    order_name = input("請輸入要查看的訂單檔案名稱（例如 order_20240601123000.csv）：")
    order_file = os.path.join(ORDER_DIR, order_name)
    
    if os.path.exists(order_file):
        try:
            with open(order_file, mode='r', encoding='utf-8') as file:
                reader = csv.reader(file)
                print("\n訂單內容：")
                for row in reader:
                    # 使用 tab 分隔呈現，使畫面更整齊
                    print("\t".join(row))
        except Exception as e:
            print(f"讀取訂單失敗: {e}")
    else:
        print("訂單檔案不存在，請重新輸入。")

def main():
    while True:
        display_menu()
        choice = input("請輸入選項 (1-7)：")
        if choice == '1':
            add_to_cart()
        elif choice == '2':
            view_cart()
        elif choice == '3':
            remove_from_cart()
        elif choice == '4':
            checkout()
        elif choice == '5':
            list_orders()
        elif choice == '6':
            view_order()
        elif choice == '7':
            print("謝謝使用，再見！")
            break
        else:
            print("無效的選項，請重新輸入。")

if __name__ == "__main__":
    main()

已建立 orders 資料夾。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

商品目錄：
御飯糰: 35 元
礦泉水: 20 元
布丁: 25 元
關東煮: 15 元
飯糰: 30 元
10 個 布丁 已加入購物車。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

目前沒有訂單。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

目前沒有訂單。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

購物車內容：
布丁 x 10
商品不在購物車中，請重新輸入。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

目前沒有訂單。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

目前沒有訂單。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

商品目錄：
御飯糰: 35 元
礦泉水: 20 元
布丁: 25 元
關東煮: 15 元
飯糰: 30 元
商品不存在，請重新輸入。

歡迎使用便利商店自助結帳系統！
請選擇操作：
1. 加入商品到購物車
2. 查看購物車
3. 移除購物車中的商品
4. 結帳
5. 列出訂單
6. 查看訂單內容
7. 離開

商品目錄：
御飯糰: 35 元
礦泉水: 20 元
布丁: 25 元
關東煮: 15 元
飯糰: 30 元
50 個 布丁 已加入購物車。

歡